In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class lora_layer(nn.Module):
    def __init__(self, in_dim, out_dim, rank=16, alpha=32, dropout=0.1):
        super().__init__()

        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.linear_a = nn.Linear(in_dim,rank)
        self.linear_b = nn.Linear(rank,out_dim)
        nn.init.kaiming_uniform_(self.linear_a.weight, math.sqrt(5))
        nn.init.zeros_(self.linear_b.weight)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.linear_b(self.linear_a(x))) * self.scaling

class LinearLora(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.linear.weight.requires_grad = False
        self.lora = lora_layer(linear.in_features, linear.out_features,rank,alpha)
    
    def forward(self,x):
        return self.linear(x) + self.lora(x)
        
def replace_linear_lora(model, r, alpha, dropout=0.1):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            lora_module = LinearLora(module,r,alpha)
            setattr(model, name, lora_module)
        else:
            replace_linear_lora(model, r, alpha, dropout)

